In [1]:
import hanlp
HanLP = hanlp.load(hanlp.pretrained.mtl.CLOSE_TOK_POS_NER_SRL_DEP_SDP_CON_ELECTRA_SMALL_ZH) # 世界最大中文语料库
# 当前支持的任务列表
tasks = list(HanLP.tasks.keys())
# 需要的任务列表
tasks_needed = ['tok/coarse', 'pos/ctb', 'dep']
print("before delete: ", tasks)
# 删除不需要的任务
for task in tasks:
    if task not in tasks_needed:
        del HanLP[task]
print("after delete: ", list(HanLP.tasks.keys()))

100% 114.3 MiB   4.1 MiB/s ETA:  0 s [=========================================]
Decompressing C:\Users\Administrator\AppData\Roaming\hanlp\mtl/close_tok_pos_ner_srl_dep_sdp_con_electra_small_20210111_124159.zip to C:\Users\Administrator\AppData\Roaming\hanlp\mtl
c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Administrator\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree.

In [7]:
from YSUtils import *

In [9]:
for txt_file in list_txt_files("./LCMC_delete_txt/"):
    lines = []
    with open(txt_file, "r", encoding="utf-8") as file:
        for line in file:
            if line == '\n':
                continue
            lines.append(line)
    doc = HanLP(lines)
    output = []
    for i, line in enumerate(lines):
        line_json = {}
        line_json["text"] = line
        line_deps_with_word = []
        line_deps = []
        for j, dep_couple in enumerate(doc["dep"][i]):
            str_with_word = doc["tok/coarse"][i][j] + " " + doc["pos/ctb"][i][j] + " " + doc["tok/coarse"][i][dep_couple[0] - 1] + " " + doc["pos/ctb"][i][dep_couple[0] - 1] + " " + dep_couple[1]
            str_dep = doc["pos/ctb"][i][j] + ' ' + doc["pos/ctb"][i][dep_couple[0] - 1] + ' ' + dep_couple[1]
            line_deps_with_word.append(str_with_word)
            line_deps.append(str_dep)
        line_json["deps_with_word"] = line_deps_with_word
        line_json["deps"] = line_deps
        output.append(line_json)
    for i in range(0, len(output), 500):
        part = output[i:i+500]  # 切片，获取 500 个元素
        json_filename = txt_file.replace(".txt", f"_{i//500}.json")  # 生成不同的文件名

        with open(json_filename, "w", encoding="utf-8") as file:
            json.dump(part, file, ensure_ascii=False, indent=4)  # 写入 JSON，格式化输出

    